In [ ]:
# Cell 1 — Imports and data load
# Run 01_explore_clean.ipynb first: it cleans the data, makes the split and writes these files.
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import cross_validate, RepeatedStratifiedKFold
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (confusion_matrix, classification_report,
                             recall_score, precision_score, f1_score,
                             roc_auc_score, accuracy_score)
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt

df = pd.read_csv("cleaned_data.csv")
with open("feature_config.json") as f:
    feature_config_in = json.load(f)

print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['at_risk'].value_counts(normalize=True)}")
print(f"\nNulls remaining: {int(df.isnull().sum().sum())}")

In [ ]:
# Cell 2 — Select model features
# Cleaning (imputation with training-split medians, one-hot encoding) happens in 01_explore_clean.ipynb.
# The model uses the 11 features listed in feature_config.json["model_features"].
FEATURES = feature_config_in['model_features']
X = df[FEATURES]
y = df['at_risk']
print(f"Feature matrix shape: {X.shape}")
print(f"Features: {FEATURES}")

In [ ]:
# Cell 3 — Train/test split (stratified 80/20, made in 01_explore_clean.ipynb)
is_train = df['split'] == 'train'
X_train, X_test = X[is_train], X[~is_train]
y_train, y_test = y[is_train], y[~is_train]
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Train at_risk rate: {y_train.mean():.3f}")
print(f"Test at_risk rate: {y_test.mean():.3f}")

In [ ]:
# Cell 4 — Baseline (majority class)
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("=== BASELINE (Majority Class) ===")
print(f"Accuracy: {accuracy_score(y_test, baseline_pred):.4f}")
print(f"Recall (at_risk=1): {recall_score(y_test, baseline_pred):.4f}")
print(f"F1 (at_risk=1): {f1_score(y_test, baseline_pred):.4f}")
print(confusion_matrix(y_test, baseline_pred))
# Expected: ~72% accuracy, 0% recall — this is the bar you must clear

## Choosing `max_depth`

A single 5-fold cross-validation is too noisy to pick the depth here: recall varies by about ±0.08 between folds, which is more than the gap between depths. Just reshuffling which students land in which fold changes the winner. So we use **repeated** stratified 5-fold CV (10 repeats = 50 fits per depth) on the training split only. The test set is never used to choose the depth.

Selection rule:
1. Consider depths **3–5**, the range where a tree is still small enough to explain to a student.
2. Find the depth with the highest mean CV **recall** on `at_risk=1`, because missing a struggling student costs more than a false alarm.
3. Pick the **shallowest** depth whose recall is within one standard error of that best (the "one-standard-error rule"). If a simpler tree is statistically as good, we prefer it.

We also print the training recall next to the validation recall. A growing gap between them is overfitting.

In [ ]:
# Cell 5 — Repeated cross-validation over max_depth (training split only)
depths = [2, 3, 4, 5, 6, 8, None]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
cv_rows = []

for d in depths:
    clf = DecisionTreeClassifier(
        max_depth=d,
        class_weight='balanced',
        random_state=42,
        min_samples_leaf=5  # prevent overfitting on small leaves
    )
    r = cross_validate(clf, X_train, y_train, cv=cv,
                       scoring=['recall', 'f1', 'roc_auc'], return_train_score=True)
    cv_rows.append({
        'max_depth': d,
        'cv_recall': r['test_recall'].mean(),
        'cv_recall_se': r['test_recall'].std() / np.sqrt(len(r['test_recall'])),
        'cv_f1': r['test_f1'].mean(),
        'cv_roc_auc': r['test_roc_auc'].mean(),
        'train_recall': r['train_recall'].mean(),
    })

cv_results = pd.DataFrame(cv_rows)
cv_results['overfit_gap'] = cv_results['train_recall'] - cv_results['cv_recall']
print(cv_results.round(4).to_string(index=False))

In [ ]:
# Cell 6 — Pick depth with the one-standard-error rule, then train the final model
CANDIDATE_DEPTHS = [3, 4, 5]  # explainable range
candidates = cv_results[cv_results['max_depth'].isin(CANDIDATE_DEPTHS)]
best = candidates.loc[candidates['cv_recall'].idxmax()]
threshold = best['cv_recall'] - best['cv_recall_se']
BEST_DEPTH = int(candidates[candidates['cv_recall'] >= threshold]['max_depth'].min())

print(f"Highest CV recall in {CANDIDATE_DEPTHS}: depth {int(best['max_depth'])} "
      f"({best['cv_recall']:.4f} ± {best['cv_recall_se']:.4f} SE)")
print(f"Shallowest depth with recall >= {threshold:.4f}: depth {BEST_DEPTH}")

model = DecisionTreeClassifier(
    max_depth=BEST_DEPTH,
    class_weight='balanced',
    random_state=42,
    min_samples_leaf=5
)
model.fit(X_train, y_train)

# Training accuracy (for overfitting check)
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"\nTrain accuracy: {train_acc:.4f}")
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Overfitting gap: {train_acc - test_acc:.4f}")
print(f"Leaves: {model.get_n_leaves()}")
# If gap > 0.10, consider reducing depth

In [ ]:
# Cell 7 — Evaluation on held-out test set
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== FINAL MODEL — TEST SET ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (at_risk=1):    {recall_score(y_test, y_pred):.4f}")
print(f"Precision (at_risk=1): {precision_score(y_test, y_pred):.4f}")
print(f"F1 (at_risk=1):        {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not At Risk', 'At Risk']))

In [ ]:
# Cell 8 — Feature importances
importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print("=== FEATURE IMPORTANCES ===")
print(importances.head(15))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
importances.head(12).plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importance (Gini)')
ax.set_title('Top 12 Feature Importances — Decision Tree')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150)
plt.show()

## Which features the model uses, and explaining one student

The tree only splits on a few of the 11 input features. The rest have **zero importance**: the tree never asks about them, so they never change a prediction.

What we do about it:
- **Keep all 11 as inputs.** The `/predict` input stays fixed. Retraining on only the used features doesn't give the same tree, and picking features from this result would mean the training data chose its own features.
- **Base each student's "why" on their decision path.** The path is the 2–3 questions the tree asked about this student. Unused features can't appear on any path, so the explanation only mentions what actually drove that student's result.
- **Export `used_features`.** The app can still give tips about sleep, study hours or flashcards, but must present them as general study advice, not as reasons the model flagged the student.

In [ ]:
# Cell 9 — Used features, the full tree, and per-student decision-path explanations
used_features = [f for f in FEATURES if model.feature_importances_[FEATURES.index(f)] > 0]
unused_features = [f for f in FEATURES if f not in used_features]
print(f"Used by the tree ({len(used_features)}): {used_features}")
print(f"Never used ({len(unused_features)}): {unused_features}\n")

tree_rules = export_text(model, feature_names=FEATURES)
print(tree_rules)


def explain_prediction(row: pd.Series) -> list[dict]:
    """Return the conditions on this student's decision path, root to leaf."""
    tree = model.tree_
    node_ids = model.decision_path(row[FEATURES].to_frame().T).indices
    steps = []
    for node in node_ids:
        if tree.children_left[node] == tree.children_right[node]:  # leaf
            continue
        feature = FEATURES[tree.feature[node]]
        threshold = float(tree.threshold[node])
        value = float(row[feature])
        steps.append({
            'feature': feature,
            'value': value,
            'threshold': round(threshold, 2),
            'direction': '<=' if value <= threshold else '>',
        })
    return steps


# Demo on one flagged and one not-flagged student from the test set
for label in (1, 0):
    idx = X_test.index[model.predict(X_test) == label][0]
    proba = model.predict_proba(X_test.loc[[idx]])[0, 1]
    print(f"Student {df.loc[idx, 'student_id']} — predicted at_risk={label} (risk {proba:.0%})")
    for s in explain_prediction(X_test.loc[idx]):
        print(f"  {s['feature']} = {s['value']:g}  ({s['direction']} {s['threshold']:g})")
    print()

In [ ]:
# Cell 10 — Export for serving
# Save model
joblib.dump(model, '../ml-service/models/tree_v1.joblib')

# Export feature list (order matters for serving)
feature_config = {
    'features': list(X.columns),
    'used_features': used_features,
    'unused_features': unused_features,
    'best_depth': BEST_DEPTH,
    'depth_selection': 'repeated stratified 5x10 CV recall, one-standard-error rule over depths 3-5',
    'cv_results': cv_results.replace({np.nan: None}).to_dict(orient='records'),
    'train_accuracy': float(train_acc),
    'test_accuracy': float(test_acc),
    'recall_at_risk': float(recall_score(y_test, y_pred)),
    'precision_at_risk': float(precision_score(y_test, y_pred)),
    'roc_auc': float(roc_auc_score(y_test, y_proba)),
    'feature_importances': importances[importances > 0].to_dict(),
    'tree_rules': tree_rules,
    'imputation_values': feature_config_in['imputation_values']
}
with open('../ml-service/models/model_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print("Exported: ml-service/models/tree_v1.joblib + model_config.json")